In [26]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ["OMP_NUM_THREADS"] = '1'
from mpi4py import MPI
import numpy as np
import quimb.tensor as qtn
import pickle
from functools import partial
import torch
import json
import autoray as ar
import symmray as sr
# ==============================================================================
from vmc_torch.experiment.vmap.vmap_utils import compute_grads, random_initial_config
from vmc_torch.experiment.vmap.vmap_models import (
    fPEPS_Model
)
from vmc_torch.experiment.vmap.vmap_modules import distributed_minres_solver, run_sampling_phase_reuse
from vmc_torch.hamiltonian_torch import spinful_Fermi_Hubbard_square_lattice_torch
from vmc_torch.experiment.tn_model import init_weights_to_zero
from vmc_torch.experiment.vmap.vmap_torch_utils import robust_svd_err_catcher_wrapper
from vmc_torch.optimizer import DecayScheduler
# ==============================================================================
import warnings
warnings.filterwarnings("ignore")
# ==============================================================================
SVD_JITTER = 1e-16
driver = None
ar.register_function('torch','linalg.svd', lambda x: robust_svd_err_catcher_wrapper(x, jitter=SVD_JITTER, driver=driver))
# ==============================================================================
COMM = MPI.COMM_WORLD
RANK = COMM.Get_rank()
SIZE = COMM.Get_size()
pwd = '/home/sijingdu/TNVMC/VMC_code/vmc_torch/vmc_torch/experiment/vmap/data'
torch.set_default_device("cpu")
torch.random.manual_seed(42 + RANK)
# ==============================================================================
# 1. Initialization & Configuration
# ==============================================================================
Lx, Ly = 4, 4
N_f = Lx * Ly - 2
D, chi = 8, -1
t, U = 1.0, 8.0

peps = sr.networks.PEPS_fermionic_rand(
    "Z2",
    Lx,
    Ly,
    D,
    phys_dim=[
        (0, 0),  # linear index 0 -> charge 0, offset 0
        (1, 1),  # linear index 1 -> charge 1, offset 1
        (1, 0),  # linear index 2 -> charge 1, offset 0
        (0, 1),  # linear index 3 -> charge 0, offset 1
    ],
    subsizes="equal",
    flat=True,
    seed=42,
    dtype='float64',
)

# ==============================================================================
# Model Configuration (Define this FIRST)
# ==============================================================================
# Put all hyperparameters for initialization here
# Note: ftn (peps) is usually too large or an object, not suitable for json storage, only record the parameters used to generate peps (Lx, Ly, etc.)
model_config = {
    'max_bond': chi,
    'embed_dim': 16,
    'attn_depth': 1,
    'attn_heads': 4,
    'nn_hidden_dim': D, #peps.nsites,
    'init_perturbation_scale': 1e-3,
    'nn_eta': 1,
    'dtype_str': 'float64',
    'jitter_svd': 0,
    'uniform_kernel': 0,
}
dtype_map = {'float64': torch.float64, 'float32': torch.float32}
model_dtype = dtype_map[model_config['dtype_str']]
init_kwargs = model_config.copy()
init_kwargs.pop('dtype_str')
# Model
# fpeps_model = Transformer_fPEPS_Model_Conv2d(
#     tn=peps,
#     dtype=model_dtype,
#     **init_kwargs
# )
fpeps_model = fPEPS_Model(
    tn=peps,
    dtype=model_dtype,
    contract_boundary_opts={
        'mode': 'mps',
        'equalize_norms': 1.0,
        'canonize': True,
    },
    **init_kwargs
)
n_params = sum(p.numel() for p in fpeps_model.parameters())
if RANK == 0: 
    print(f'Model Params: {n_params}')

# Hamiltonian
H = spinful_Fermi_Hubbard_square_lattice_torch(
    Lx, Ly, t, U, N_f, pbc=False, n_fermions_per_spin=(N_f//2, N_f//2), no_u1_symmetry=False,
)

Model Params: 41472


In [27]:
import quimb as qu
params, skeleton = qtn.pack(peps)
ftn_params_flat, ftn_params_pytree = qu.utils.tree_flatten(params, get_ref=True)
ftn_params_shape = [p.shape for p in ftn_params_flat]

In [28]:
ftn_params_pytree, ftn_params_shape

({0: {'blocks': Leaf},
  1: {'blocks': Leaf},
  2: {'blocks': Leaf},
  3: {'blocks': Leaf},
  4: {'blocks': Leaf},
  5: {'blocks': Leaf},
  6: {'blocks': Leaf},
  7: {'blocks': Leaf},
  8: {'blocks': Leaf},
  9: {'blocks': Leaf},
  10: {'blocks': Leaf},
  11: {'blocks': Leaf},
  12: {'blocks': Leaf},
  13: {'blocks': Leaf},
  14: {'blocks': Leaf},
  15: {'blocks': Leaf}},
 [(4, 4, 4, 2),
  (8, 4, 4, 4, 2),
  (8, 4, 4, 4, 2),
  (4, 4, 4, 2),
  (8, 4, 4, 4, 2),
  (16, 4, 4, 4, 4, 2),
  (16, 4, 4, 4, 4, 2),
  (8, 4, 4, 4, 2),
  (8, 4, 4, 4, 2),
  (16, 4, 4, 4, 4, 2),
  (16, 4, 4, 4, 4, 2),
  (8, 4, 4, 4, 2),
  (4, 4, 4, 2),
  (8, 4, 4, 4, 2),
  (8, 4, 4, 4, 2),
  (4, 4, 4, 2)])

In [43]:
import torch
import torch.nn as nn
import quimb as qu
import quimb.tensor as qtn

class LoRA_fPEPS_Model(nn.Module):
    def __init__(self, tn, max_bond, lora_rank=8, dtype=torch.float64, compile=False, contract_boundary_opts={}, **kwargs):
        super().__init__()
        
        # 1. Basic Configuration
        self.dtype = dtype
        self.chi = max_bond
        self.contract_boundary_opts = contract_boundary_opts
        self.lora_rank = lora_rank
        
        # 2. Extract Structure
        params, skeleton = qtn.pack(tn)
        self.skeleton = skeleton
        params_flat, params_pytree = qu.utils.tree_flatten(params, get_ref=True)
        self.params_pytree = params_pytree

        # 3. Register Fixed Tensors as BUFFERS (关键修改)
        # 使用 register_buffer，它们不会出现在 model.parameters() 中
        # 这保证了你的 optimizer step 更新参数时，维度只包含 LoRA 参数
        self.fixed_tensors = [] 
        for i, x in enumerate(params_flat):
            # 命名为 fixed_0, fixed_1 以便随 state_dict 保存
            name = f"fixed_tensor_{i}"
            t = torch.as_tensor(x, dtype=self.dtype).detach()
            self.register_buffer(name, t)
            self.fixed_tensors.append(t) # 保留一个 list 引用方便后续 zip 遍历

        # 4. Register LoRA Parameters
        self.lora_sites = nn.ModuleList()
        for tensor_data in params_flat:
            shape = tensor_data.shape
            n_blocks = shape[0]
            phys_dims = shape[1:] 
            
            factors = nn.ParameterList()
            for dim_size in phys_dims:
                # Batched CP Factors
                factor = torch.randn(n_blocks, lora_rank, dim_size, dtype=self.dtype) * 0.01
                factors.append(nn.Parameter(factor))
            self.lora_sites.append(factors)

        # 5. Vmap setup
        self._vmapped_amplitude = torch.vmap(
            self.amplitude,
            in_dims=(0, None), # x is batched, params is shared
            randomness='different',
        )
        if compile:
            self._vmapped_amplitude = torch.compile(
                self._vmapped_amplitude, fullgraph=False, mode="default"
            )

    @property
    def params(self):
        """
        关键属性：为 compute_grads 提供纯净的参数列表。
        返回一个 List of Lists (PyTree)，只包含 LoRA 参数。
        """
        # self.lora_sites 是 ModuleList，里面是 ParameterList
        # 我们将其解包成纯粹的 Tensor List 结构，供 torch.func 使用
        return [[p for p in site] for site in self.lora_sites]

    def _reconstruct_params(self, lora_params_structure=None):
        """
        支持函数式注入参数。
        如果 lora_params_structure 传入了 (来自 vmap)，则使用它；
        否则使用模型自带的 self.lora_sites。
        """
        # 决定使用哪组参数
        if lora_params_structure is None:
            # 默认情况（非函数式调用），使用自身参数
            iter_params = self.lora_sites
        else:
            # 函数式调用（compute_grads），使用传入的 params
            iter_params = lora_params_structure

        current_params_flat = []
        
        # 这里的 zip 会自动跳过不可训练的 fixed_tensors 的梯度计算，
        # 因为它们被 torch.func 视为 captured constants
        for fixed_t, factors in zip(self.fixed_tensors, iter_params):
            n_phys_dims = len(fixed_t.shape) - 1 
            input_subs = [f"xr{chr(97+i)}" for i in range(n_phys_dims)] 
            output_sub = "x" + "".join([chr(97+i) for i in range(n_phys_dims)])
            eq = f"{','.join(input_subs)}->{output_sub}"
            
            # factors 在这里是一个 tensor list (如果是函数式调用)
            # *factors 解包支持 list 输入
            delta_t = torch.einsum(eq, *factors)
            current_params_flat.append(fixed_t + delta_t)
            
        return qu.utils.tree_unflatten(current_params_flat, self.params_pytree)

    def amplitude(self, x, params):
        # params 这里是 reconstruct 出来的 full tensor dict，无需改动
        tn = qtn.unpack(params, self.skeleton)
        amp = tn.isel({tn.site_ind(site): x[i] for i, site in enumerate(tn.sites)})
        if self.chi > 0:
            amp.contract_boundary_from_xmin_(max_bond=self.chi, cutoff=0.0, xrange=[0, amp.Lx//2-1], **self.contract_boundary_opts)
            amp.contract_boundary_from_xmax_(max_bond=self.chi, cutoff=0.0, xrange=[amp.Lx//2, amp.Lx-1], **self.contract_boundary_opts)
        return amp.contract()
    
    def vamp(self, x, params=None):
        """
        关键修改：增加 params 可选参数，适配 compute_grads 的 single_sample_amp_func(x, p) 调用。
        """
        # 1. 动态重构参数 (传入的 params 是 LoRA factors)
        combined_params_pytree = self._reconstruct_params(lora_params_structure=params)
        
        # 2. 计算振幅 (传入的是 Full Tensors)
        return self._vmapped_amplitude(x, combined_params_pytree)

    def forward(self, x):
        return self.vamp(x)
    
test_lora_model = LoRA_fPEPS_Model(
    tn=peps,
    dtype=model_dtype,
    max_bond=chi,
    lora_rank=8,
)

In [44]:
def print_model_params_breakdown(model, model_name="Model"):
    """
    Counts and prints trainable vs. fixed parameters.
    """
    # 1. Calculate total parameters
    total_params = sum(p.numel() for p in model.parameters())
    
    # 2. Calculate trainable parameters (requires_grad=True)
    # These are your LoRA factors
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # 3. Calculate fixed parameters (requires_grad=False)
    # These are your fixed background tensors
    fixed_params = total_params - trainable_params
    
    print(f"--- {model_name} Statistics ---")
    print(f"Total Parameters:     {total_params:,}")
    print(f"Trainable (LoRA):     {trainable_params:,}  ({trainable_params/total_params:.2%})")
    print(f"Fixed (Background):   {fixed_params:,}")
    print("----------------------------\n")
    
    return trainable_params, fixed_params

# Usage example:
# For your standard full-rank PEPS (if everything is trainable)
# fpeps_trainable, _ = print_model_params_breakdown(fpeps_model, "Original fPEPS")

# For your LoRA model
# lora_trainable, lora_fixed = print_model_params_breakdown(test_lora_model, "LoRA fPEPS")

print_model_params_breakdown(test_lora_model, "LoRA fPEPS")
print_model_params_breakdown(fpeps_model, "Original fPEPS")

--- LoRA fPEPS Statistics ---
Total Parameters:     17,664
Trainable (LoRA):     17,664  (100.00%)
Fixed (Background):   0
----------------------------

--- Original fPEPS Statistics ---
Total Parameters:     41,472
Trainable (LoRA):     41,472  (100.00%)
Fixed (Background):   0
----------------------------



(41472, 0)

In [45]:
# Init State
B = 10
fxs = torch.stack([random_initial_config(N_f, peps.nsites) for _ in range(B)]).to(torch.long)
test_lora_model(fxs)

tensor([ 7.3316e+07,  1.4534e+08,  2.7550e+08, -1.1267e+08,  1.7378e+08,
         8.3841e+07, -4.2113e+07, -2.9370e+07,  1.2693e+08, -4.2374e+08],
       dtype=torch.float64, grad_fn=<MulBackward0>)

In [46]:
fpeps_model(fxs)

tensor([ 7.3313e+07,  1.4534e+08,  2.7550e+08, -1.1267e+08,  1.7378e+08,
         8.3841e+07, -4.2117e+07, -2.9374e+07,  1.2693e+08, -4.2374e+08],
       dtype=torch.float64, grad_fn=<MulBackward0>)

In [63]:
import torch
import torch.nn as nn
import quimb as qu
import quimb.tensor as qtn

class FactorGenerator(nn.Module):
    """
    A lightweight MLP that generates LoRA factors based on the global configuration x.
    """
    def __init__(self, input_dim, output_shapes, hidden_dim=64, dtype=torch.float64):
        super().__init__()
        self.output_shapes = output_shapes # List of (N_b, Rank, Dim)
        
        # Total number of elements to generate across all factors
        self.total_elements = sum(torch.prod(torch.tensor(s)).item() for s in output_shapes)
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim, dtype=dtype),
            nn.Tanh(), # Tanh is often better for wavefunctions than ReLU
            nn.Linear(hidden_dim, int(self.total_elements), dtype=dtype)
        )

    def forward(self, x):
        # x shape: (Batch, N_sites)
        flat_outputs = self.net(x.to(next(self.parameters()).dtype)) # (Batch, Total_Elements)
        
        # Split and reshape into the required factor shapes
        all_factors = []
        cursor = 0
        for shape in self.output_shapes:
            num_el = int(torch.prod(torch.tensor(shape)).item())
            # Reshape to (Batch, N_b, Rank, Dim)
            all_factors.append(flat_outputs[:, cursor:cursor+num_el].view(-1, *shape))
            cursor += num_el
        return all_factors

class LoRA_NNfPEPS_Model(nn.Module):
    def __init__(self, tn, max_bond, lora_rank=8, hidden_dim=64, dtype=torch.float64, **kwargs):
        super().__init__()
        self.dtype = dtype
        self.chi = max_bond
        self.lora_rank = lora_rank
        self.n_sites = len(tn.sites)
        
        # 1. Extract TN structure
        params, skeleton = qtn.pack(tn)
        self.skeleton = skeleton
        params_flat, params_pytree = qu.utils.tree_flatten(params, get_ref=True)
        self.params_pytree = params_pytree

        # 2. Register Fixed Tensors as BUFFERS (Keep parameters minimal)
        self.fixed_tensors = []
        for i, x in enumerate(params_flat):
            self.register_buffer(f"fixed_tensor_{i}", torch.as_tensor(x, dtype=self.dtype).detach())
            self.fixed_tensors.append(getattr(self, f"fixed_tensor_{i}"))

        # 3. Prepare Shapes for the Factor Generator
        # Each tensor site has multiple factors (one per leg)
        self.all_factor_shapes = []
        self.site_factor_counts = []
        for tensor_data in params_flat:
            shape = tensor_data.shape # (N_b, D/2, ..., d/2)
            n_blocks = shape[0]
            phys_dims = shape[1:]
            self.site_factor_counts.append(len(phys_dims))
            for dim_size in phys_dims:
                self.all_factor_shapes.append((n_blocks, lora_rank, dim_size))

        # 4. Neural Factor Generator
        # This is the ONLY part with trainable parameters (besides the optional NN weights)
        self.generator = FactorGenerator(
            input_dim=self.n_sites, 
            output_shapes=self.all_factor_shapes, 
            hidden_dim=hidden_dim, 
            dtype=dtype
        )

    @property
    def params(self):
        """
        Returns the generator's parameters for compute_grads.
        """
        return list(self.generator.parameters())

    def _reconstruct_params_batched(self, x, generator_params=None):
        """
        Reconstruct tensors for each configuration in the batch.
        Output shape for each tensor: (Batch, N_b, D/2, ..., d/2)
        """
        # If generator_params is provided (from torch.func), we use functional call
        if generator_params is not None:
            from torch.func import functional_call
            param_dict = {name: p for name, p in zip([n for n, _ in self.generator.named_parameters()], generator_params)}
            all_generated_factors = functional_call(self.generator, param_dict, (x,))
        else:
            all_generated_factors = self.generator(x)

        current_params_flat_batched = []
        factor_cursor = 0
        
        for fixed_t in self.fixed_tensors:
            num_factors = len(fixed_t.shape) - 1
            factors = all_generated_factors[factor_cursor : factor_cursor + num_factors]
            factor_cursor += num_factors
            
            # --- CORRECTED EINSUM LOGIC ---
            n_phys_dims = len(fixed_t.shape) - 1
            
            # We use UPPERCASE for structural indices to avoid collision with 'a','b','c'...
            # Z: Batch dimension (from x)
            # S: Symmetry Block dimension (N_b)
            # R: LoRA Rank dimension
            # a, b, c... : Physical/Virtual dimensions
            
            # Input subscripts: "ZSRa", "ZSRb", "ZSRc" ...
            input_subs = [f"ZSR{chr(97+i)}" for i in range(n_phys_dims)]
            
            # Output subscript: "ZSabcd..." 
            # Z and S are preserved (batch dims), R is contracted (summed over).
            output_sub = "ZS" + "".join([chr(97+i) for i in range(n_phys_dims)])
            
            eq = f"{','.join(input_subs)}->{output_sub}"
            
            # delta_t shape: (Batch, N_b, D/2, D/2, ..., d/2)
            delta_t = torch.einsum(eq, *factors)
            
            # Broadcast the fixed_t background across the Batch dimension
            # fixed_t: (N_b, ...) -> unsqueeze -> (1, N_b, ...)
            current_params_flat_batched.append(fixed_t.unsqueeze(0) + delta_t)
            
        return current_params_flat_batched

    def amplitude_single(self, x_single, params_single):
        """
        Compute amplitude for a single config with its specific reconstructed tensor.
        """
        # Unflatten the dict/pytree for quimb
        params_dict = qu.utils.tree_unflatten(params_single, self.params_pytree)
        tn = qtn.unpack(params_dict, self.skeleton)
        
        amp = tn.isel({tn.site_ind(site): x_single[i] for i, site in enumerate(tn.sites)})
        if self.chi > 0:
            # Boundary contraction
            amp.contract_boundary_from_xmin_(max_bond=self.chi, xrange=[0, amp.Lx//2-1])
            amp.contract_boundary_from_xmax_(max_bond=self.chi, xrange=[amp.Lx//2, amp.Lx-1])
        return amp.contract()

    def vamp(self, x, params=None):
        """
        Overridden vamp to handle config-dependent tensors.
        """
        # 1. Generate tensors for each x in the batch
        # batched_tensors: List of Tensors, each (Batch, N_b, ...)
        batched_tensors = self._reconstruct_params_batched(x, generator_params=params)
        
        # 2. Map amplitude calculation over the batch
        # We need to slice batched_tensors along the 0-th dimension inside vmap
        def compute_single(x_i, *tensors_i):
            return self.amplitude_single(x_i, tensors_i)

        return torch.vmap(compute_single)(x, *batched_tensors)

    def forward(self, x):
        return self.vamp(x)

test_nn_lora_fpeps_model = LoRA_NNfPEPS_Model(
    tn=peps,
    dtype=model_dtype,
    max_bond=chi,
    lora_rank=4,
    hidden_dim=4,
)

In [64]:
print_model_params_breakdown(test_nn_lora_fpeps_model, "Neural LoRA fPEPS")

--- Neural LoRA fPEPS Statistics ---
Total Parameters:     44,228
Trainable (LoRA):     44,228  (100.00%)
Fixed (Background):   0
----------------------------



(44228, 0)

In [65]:
test_nn_lora_fpeps_model(fxs)

tensor([ 1.4355e+08,  1.6874e+08,  2.8383e+08, -2.7967e+07,  1.8142e+08,
         1.2848e+08,  1.4041e+08, -2.3812e+08,  1.0708e+08, -3.8101e+08],
       dtype=torch.float64, grad_fn=<MulBackward0>)

In [ ]:
from models.LoRA_models import *

class BasefPEPSBackflowModel(nn.Module):
    """
    Base Class: Handles fPEPS parameter management, vmap contraction logic, 
    and general forward pass flow.
    
    Subclasses must define `self.nn_backflow` in `__init__` and call 
    `self.finish_initialization()`.
    """
    def __init__(
        self,
        tn,
        max_bond,
        nn_eta,
        dtype=torch.float64,
        debug_file=None,
        contract_boundary_opts={}
    ):
        super().__init__()
        self.contract_boundary_opts = contract_boundary_opts
        self.dtype = dtype
        self.debug_file = debug_file
        self.chi = max_bond
        self.nn_eta = nn_eta

        # --- 1. Tensor Network Setup (Common) ---
        # Extract raw arrays and skeleton
        params, skeleton = qtn.pack(tn)
        self.skeleton = skeleton
        self.skeleton.exponent = 0 
        
        # Flatten TN parameters into a single list and a PyTree structure
        ftn_params_flat, ftn_params_pytree = qu.utils.tree_flatten(params, get_ref=True)
        self.ftn_params_pytree = ftn_params_pytree

        # Register TN parameters as a ParameterList so optimizer can see them
        self.ftn_params = torch.nn.ParameterList([
            torch.as_tensor(x, dtype=self.dtype) for x in ftn_params_flat
        ])
        
        # Metadata for reconstruction inside vmap
        self.ftn_params_shape = [p.shape for p in self.ftn_params]
        self.ftn_params_sizes = [p.numel() for p in self.ftn_params] 
        self.ftn_params_length = sum(self.ftn_params_sizes)

        # Placeholders for Child class to fill
        self.nn_backflow = None 
        self.nn_backflow_generator = None 
        self.nn_param_names = None
        self.params = None

        # vamp func
        self._vamp = torch.vmap(self.tn_contraction, in_dims=(0, None, 0), randomness='different')

    def finish_initialization(self, init_scale=1e-5):
        """
        Must be called by the subclass after `self.nn_backflow` is defined.
        It registers all parameters and initializes weights.
        """
        if self.nn_backflow is None:
            raise ValueError("Child class must define self.nn_backflow before calling finish_initialization")

        # 1. Register NN parameter names for functional_call
        self.nn_param_names = [name for name, _ in self.nn_backflow.named_parameters()]
        
        # 2. Combine all params into one list for the optimizer
        # Order: [TN Params ... NN Params]
        self.params = nn.ParameterList(list(self.ftn_params) + list(self.nn_backflow.parameters()))
        
        # 3. Initialize perturbation weights to be small
        self._init_weights_for_perturbation(scale=init_scale)

    def _init_weights_for_perturbation(self, scale=1e-5):
        """
        Delegates initialization to the backflow generator (Generic Interface Pattern).
        """
        target_module = self.nn_backflow_generator if self.nn_backflow_generator else self.nn_backflow
        
        if hasattr(target_module, 'initialize_output_scale'):
            target_module.initialize_output_scale(scale)
        else:
            # Fallback for simple structures
            print(f"Warning: {type(target_module).__name__} does not implement 'initialize_output_scale'.")
            # Try to guess standard layers (Sequential/ModuleList)
            last_layer = None
            if isinstance(target_module, nn.Sequential):
                last_layer = target_module[-1]
            
            if last_layer and hasattr(last_layer, 'weight'):
                 print(f" -> Initializing last layer {type(last_layer).__name__} with scale {scale}")
                 torch.nn.init.normal_(last_layer.weight, mean=0.0, std=scale)
                 if last_layer.bias is not None: 
                     torch.nn.init.zeros_(last_layer.bias)

    def tn_contraction(self, x, ftn_params, nn_output):
        """ 
        Core logic for vmap:
        1. Reconstruct TN parameters from vector.
        2. Add NN backflow correction.
        3. Pack into Quimb TN.
        4. Perform contraction.
        """
        # 1. Reconstruct the vector
        ftn_params_vector = nn.utils.parameters_to_vector(ftn_params)
        
        # 2. Add backflow (NN correction)
        # nn_output is a single sample correction vector
        nnftn_params_vector = ftn_params_vector + self.nn_eta * nn_output
        
        # 3. Unpack to PyTree (list of tensors)
        nnftn_params = []
        pointer = 0
        for shape in self.ftn_params_shape:
            length = torch.prod(torch.tensor(shape)).item()
            nnftn_params.append(nnftn_params_vector[pointer:pointer+length].view(shape))
            pointer += length
        
        # Restore dictionary structure and unpack to Quimb TN
        nnftn_params = qu.utils.tree_unflatten(nnftn_params, self.ftn_params_pytree)
        tn = qtn.unpack(nnftn_params, self.skeleton)
        
        # 4. Contraction
        # Note: x here is a single sample (tensor of indices)
        # Select tensors based on configuration x
        amp = tn.isel({tn.site_ind(site): x[i] for i, site in enumerate(tn.sites)})
        
        # Contract boundary environments if max_bond is set
        if self.chi > 0:
            amp.contract_boundary_from_ymin_(max_bond=self.chi, cutoff=0.0, yrange=[0, amp.Ly//2-1], **self.contract_boundary_opts)
            amp.contract_boundary_from_ymax_(max_bond=self.chi, cutoff=0.0, yrange=[amp.Ly//2, amp.Ly-1], **self.contract_boundary_opts)
        
        return amp.contract()

    def vamp(self, x, params):
        """
        Batched computation:
        1. functional_call to compute Backflow (Batch Mode).
        2. vmap to compute TN Contraction.
        """
        # 1. Split params into TN part and NN part
        n_ftn = len(self.ftn_params)
        ftn_params = params[:n_ftn]
        nn_params = params[n_ftn:]
        
        # Reconstruct NN param dict for functional_call
        nn_params_dict = dict(zip(self.nn_param_names, nn_params))

        # 2. Compute Backflow (Batch Mode)
        # nn_backflow handles the logic (Global Attn, Local Conv, or Independent Cluster)
        batch_nn_outputs = torch.func.functional_call(self.nn_backflow, nn_params_dict, x)

        # 3. vmap TN Contraction
        # Map over x (dim 0) and nn_outputs (dim 0)
        # We do NOT map over ftn_params (None)
        amps = self._vamp(x, ftn_params, batch_nn_outputs)
            
        return amps

    def forward(self, x):
        # Ensure inputs are long type for embeddings/indexing
        if x.dtype != torch.long:
             x = x.to(torch.long)
        
        # Forward pass wraps vamp with optional jitter context
        return self.vamp(x, self.params)

class LoRA_LocalClusterGenerator(nn.Module):
    """
    Replaces the simple MLP FactorGenerator.
    Manages N independent LocalSiteNetworks to generate LoRA factors.
    """
    def __init__(self, tn, lora_rank, embed_dim, attn_heads, hidden_dim, radius=1, dtype=torch.float64):
        super().__init__()
        self.dtype = dtype
        self.Lx = tn.Lx
        self.Ly = tn.Ly
        
        # 1. Calculate Receptive Fields
        rf_dict = get_receptive_field_2d(self.Lx, self.Ly, radius)
        n_sites = self.Lx * self.Ly
        self.rf_indices = [rf_dict[i] for i in range(n_sites)]
        
        # 2. Extract on-site tensor shape information for LoRA factors
        # We need to know how many LoRA parameters each site requires.
        ftn_params, _ = qtn.pack(tn)
        ftn_params_flat, _ = qu.utils.tree_flatten(ftn_params, get_ref=True)
        # Record on-site tensor block shapes
        self.ftn_params_shape = [p.shape for p in ftn_params_flat] # p.shape (N_b, D/2, ..., D/2, d/2)
        
        self.site_configs = [] # Stores (output_dim, list_of_factor_shapes) for each site
        
        for tensor_data in ftn_params_flat:
            shape = tensor_data.shape # (N_b, D/2, ..., d/2)
            n_blocks = shape[0]
            phys_dims = shape[1:]
            
            # Calculate total elements needed for LoRA factors at this site
            # Each factor shape is (N_b, Rank, Dim)
            site_factor_shapes = []
            total_site_params = 0
            
            for dim_size in phys_dims:
                f_shape = (n_blocks, lora_rank, dim_size)
                site_factor_shapes.append(f_shape)
                total_site_params += int(torch.prod(torch.tensor(f_shape)).item())
            
            self.site_configs.append({
                "output_dim": total_site_params,
                "fshapes": site_factor_shapes
            })

        # 3. Create N Independent Networks
        self.site_networks = nn.ModuleList()
        for i in range(n_sites):
            n_neighbors = len(self.rf_indices[i])
            output_dim = self.site_configs[i]["output_dim"]
            
            net = LocalSiteNetwork(
                n_neighbors=n_neighbors,
                num_classes=tn.phys_dim(), # e.g. 2 for Spin-1/2, 4 for Fermions
                embed_dim=embed_dim,
                attention_heads=attn_heads,
                hidden_dim=hidden_dim,
                output_dim=output_dim,
                dtype=dtype
            )
            self.site_networks.append(net)
    
    def initialize_output_scale(self, scale):
        print(f" -> [Init] LocalClusterBackflow: Clamping output weights to scale {scale}")
        for net in self.site_networks:
            # The last layer of the MLP is at index 2
            last_layer = net.mlp[-1]
            torch.nn.init.normal_(last_layer.weight, mean=0.0, std=scale)
            if last_layer.bias is not None:
                torch.nn.init.zeros_(last_layer.bias)
                    

    def forward(self, x):
        # x: (Batch, N_sites) Int/Long
        nn_outputs = []
        # Loop over each site network
        for i, net in enumerate(self.site_networks):
            # 1. Get local neighborhood
            neighbors = self.rf_indices[i]
            x_local = x[:, neighbors] # (Batch, n_neighbors)
            
            # 2. Run local network -> (Batch, Total_Site_Params)
            site_output_flat = net(x_local)
            
            # 3. Split and Reshape into individual factors
            cursor = 0
            site_fshapes = self.site_configs[i]["fshapes"]
            site_ts_shapes = self.ftn_params_shape[i] # (N_b, D/2, ..., d/2)
            
            bfactors = []
            
            for shape in site_fshapes:
                num_el = int(torch.prod(torch.tensor(shape)).item()) # N_b * Rank * Dim
                bfactor = site_output_flat[:, cursor : cursor+num_el].view(-1, *shape) # (Batch, N_b, LoRA_Rank, Dim)
                bfactors.append(bfactor)
                cursor += num_el
            
            # einsum logic to reshape bfactor to match the corresponding tensor leg
            n_factorized_dims = len(site_ts_shapes) - 1
            input_subs = [f'BSR{chr(97+i)}' for i in range(n_factorized_dims)] # e.g. BSRa, BSRb, ...
            output_sub = 'BS' + ''.join([chr(97+i) for i in range(n_factorized_dims)]) # e.g. Bsab..
            eq = f"{','.join(input_subs)}->{output_sub}"
            delta_t = torch.einsum(eq, *bfactors) # (Batch, N_b, D/2, D/2, ..., d/2)
            # flatten to (Batch, Total_Site_Params) for addition to the original tensor vector
            delta_t_flat = delta_t.view(delta_t.size(0), -1) # (Batch, Total_Site_Params)
            nn_outputs.append(delta_t_flat)
        
        nn_outputs = torch.cat(nn_outputs, dim=1) # (Batch, ftn_params_length)
            
        return nn_outputs # shape (Batch, ftn_params_length)

# ==============================================================================
# Main Model Integration
# ==============================================================================

class LoRA_NNfPEPS_Model_Cluster(BasefPEPSBackflowModel):
    
    """
    Subclass B: Local Cluster Backflow.
    Totally independent neural networks per site. No global attention.
    Input for each site is determined strictly by its receptive field.
    """
    def __init__(
        self,
        tn,
        max_bond,
        nn_eta,
        nn_hidden_dim,
        embed_dim,
        attn_heads,
        radius=1,
        lora_rank=4,
        init_perturbation_scale=1e-5,
        dtype=torch.float64,
        **kwargs,
    ):
        # 1. Call Base Init
        super().__init__(
            tn,
            max_bond,
            nn_eta,
            dtype,
            kwargs.get("debug_file"),
            contract_boundary_opts=kwargs.get("contract_boundary_opts", {}),
        )
        
        self.lora_rank = lora_rank

        # 2. Define NN Architecture (Local & Independent)
        self.nn_backflow_generator = LoRA_LocalClusterGenerator(
            tn=tn,
            radius=radius,
            embed_dim=embed_dim,
            attn_heads=attn_heads,
            hidden_dim=nn_hidden_dim,
            dtype=self.dtype,
            lora_rank=self.lora_rank,
        )
        
        # Direct assignment (no global attn prepended)
        self.nn_backflow = self.nn_backflow_generator

        # 3. Finalize initialization
        self.finish_initialization(init_perturbation_scale)

In [33]:
import symmray as sr
from vmc_torch.experiment.vmap.vmap_utils import random_initial_config
Lx, Ly = 4, 2
D = 4
chi = -1
dtype = torch.float64
peps = sr.networks.PEPS_fermionic_rand(
    "Z2",
    Lx,
    Ly,
    D,
    phys_dim=[
        (0, 0),  # linear index 0 -> charge 0, offset 0
        (1, 1),  # linear index 1 -> charge 1, offset 1
        (1, 0),  # linear index 2 -> charge 1, offset 0
        (0, 1),  # linear index 3 -> charge 0, offset 1
    ],
    subsizes="equal",
    flat=True,
    seed=42,
    dtype=str(dtype).split(".")[-1],
)
fxs = torch.stack([random_initial_config(Lx*Ly//2, Lx*Ly, seed=42+_) for _ in range(10)])
fpeps_model = LoRA_NNfPEPS_Model_Cluster(
    tn=peps,
    max_bond=chi,
    dtype=dtype,
    nn_eta=1.0,
    nn_hidden_dim=16,
    embed_dim=16,
    attn_heads=4,
    radius=1,
    lora_rank=1,
    init_perturbation_scale=1e-3,
)
fpeps_model.ftn_params_shape
fpeps_model(fxs)

 -> [Init] LocalClusterBackflow: Clamping output weights to scale 0.001


tensor([238.1628, -55.3991, -24.4074,  23.5108,  27.2533, -34.7301,  30.4005,
         34.1233, -67.2701,  60.9541], dtype=torch.float64,
       grad_fn=<MulBackward0>)